In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device:    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"PyTorch:        {torch.__version__}")

CUDA available: True
CUDA device:    NVIDIA H100-80C
PyTorch:        2.9.1+cu128


In [2]:
cd

/slurm/homes/pchandra


## Connecting to Neo4J

In [2]:
from neo4j import GraphDatabase
from pathlib import Path
import csv

PROJECT_ROOT = Path.home() / "xai-knowledge-graph"
TRIPLES_DIR  = PROJECT_ROOT / "data" / "triples"
TRIPLES_DIR.mkdir(parents=True, exist_ok=True)

def load_creds(path=str(Path.home() / "aura_cred.txt")):
    creds = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                creds[k.strip()] = v.strip()
    return creds

creds  = load_creds()
driver = GraphDatabase.driver(creds["NEO4J_URI"], auth=(creds["NEO4J_USERNAME"], creds["NEO4J_PASSWORD"]))
DB     = creds["NEO4J_DATABASE"]
print("Connected to Neo4j")

Connected to Neo4j


In [4]:
triples = []

with driver.session(database=DB) as s:
    # AUTHORED: (author, authored, paper)
    print("Extracting AUTHORED...")
    for r in s.run("MATCH (a:Author)-[:AUTHORED]->(p:Paper) RETURN a.name AS h, p.arxiv_id AS t"):
        triples.append((f"author:{r['h']}", "authored", f"paper:{r['t']}"))

    # CITES: (paper, cites, paper)
    print("Extracting CITES...")
    for r in s.run("MATCH (a:Paper)-[:CITES]->(b:Paper) RETURN a.arxiv_id AS h, b.arxiv_id AS t"):
        triples.append((f"paper:{r['h']}", "cites", f"paper:{r['t']}"))

    # ABOUT: (paper, about, topic)
    print("Extracting ABOUT...")
    for r in s.run("MATCH (p:Paper)-[:ABOUT]->(tp:Topic) RETURN p.arxiv_id AS h, tp.name AS t"):
        triples.append((f"paper:{r['h']}", "about", f"topic:{r['t']}"))

    # PUBLISHED_IN: (paper, published_in, venue)
    print("Extracting PUBLISHED_IN...")
    for r in s.run("MATCH (p:Paper)-[:PUBLISHED_IN]->(v:Venue) RETURN p.arxiv_id AS h, v.name AS t"):
        triples.append((f"paper:{r['h']}", "published_in", f"venue:{r['t']}"))

print(f"\n✓ Total triples extracted: {len(triples):,}")

Extracting AUTHORED...
Extracting CITES...
Extracting ABOUT...
Extracting PUBLISHED_IN...

✓ Total triples extracted: 34,451


In [5]:
triples_path = TRIPLES_DIR / "xai_kg.tsv"
with open(triples_path, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    for triple in triples:
        writer.writerow(triple)
print(f"Saved {len(triples):,} triples to {triples_path}")
print(f"File size: {triples_path.stat().st_size / 1024:.1f} KB")

Saved 34,451 triples to /slurm/homes/pchandra/xai-knowledge-graph/data/triples/xai_kg.tsv
File size: 1581.3 KB


In [6]:
from collections import Counter

# Count by relation type
rel_counts = Counter(t[1] for t in triples)
print("Triples per relation:")
for rel, n in rel_counts.most_common():
    print(f"  {rel:<15} {n:>7,}")

# Count unique entities
unique_entities = set()
for h, _, t in triples:
    unique_entities.add(h)
    unique_entities.add(t)
print(f"\nUnique entities: {len(unique_entities):,}")
print(f"Unique relations: {len(rel_counts)}")

Triples per relation:
  authored         17,119
  about            11,396
  published_in      3,299
  cites             2,637

Unique entities: 18,603
Unique relations: 4
